# Accent Fleet Phase 3 — End-to-End Demo

This notebook walks the entire stream-ready pipeline on the live staging data:

1. **Profile** staging — what are we working with?
2. **Bootstrap** — create schemas, state tables, static dimensions.
3. **Incremental load** — run one window.
4. **Feature mart** — inspect the 35-feature table.
5. **Risk scoring** — device-level composite scores.
6. **Stream simulation** — replay historical path rows as if they were live.
7. **Quality & freshness** — run the validation suite.

> Every code cell uses the `accent_fleet` package. If you understand this notebook, you understand the pipeline.


## 0 · Environment


In [ ]:
from pathlib import Path
import sys
# Make the package importable when this notebook sits in notebooks/
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import polars as pl
import pandas as pd
from sqlalchemy import text

from accent_fleet.config import settings, load_pipeline_config
from accent_fleet.db import get_engine

print('DB:', settings().pg_database, '@', settings().pg_host)
print('Env:', settings().pipeline_env)


## 1 · Profile staging
A quick landscape of what we inherit from upstream.


In [ ]:
engine = get_engine()
profile_sql = '''
SELECT 'path'               AS src, COUNT(*) AS rows, MIN(begin_path_time) AS min_t, MAX(begin_path_time) AS max_t FROM staging.path
UNION ALL SELECT 'stop',           COUNT(*), MIN(stop_start), MAX(stop_start) FROM staging.stop
UNION ALL SELECT 'rep_overspeed',  COUNT(*), MIN(begin_path_time), MAX(begin_path_time) FROM staging.rep_overspeed
UNION ALL SELECT 'notification',   COUNT(*), MIN(created_at), MAX(created_at) FROM staging.notification
ORDER BY rows DESC;
'''
profile = pd.read_sql(profile_sql, engine)
profile

### 1.1 Data-quality preview — why the cleaning rules matter


In [ ]:
dq_sql = '''
SELECT
  COUNT(*) FILTER (WHERE begin_path_time < '2019-10-01')     AS c1_epoch_errors,
  COUNT(*) FILTER (WHERE path_duration   <= 0)                 AS c2_bad_durations,
  COUNT(*) FILTER (WHERE distance_driven <= 0)                 AS c3_zero_distance,
  COUNT(*) FILTER (WHERE fuel_used < 0 OR fuel_used > 500)     AS c4_fuel_outliers,
  COUNT(*) FILTER (WHERE max_speed > 200)                      AS c5_speed_over_cap
FROM staging.path
'''
pd.read_sql(dq_sql, engine)

## 2 · Bootstrap
Run once. Creates the three schemas, the watermark table, the run log, and all six dimensions.


In [ ]:
from accent_fleet.pipeline import bootstrap_flow
bootstrap_flow()

In [ ]:
# Verify the state tables are ready
pd.read_sql('SELECT layer, table_name, last_event_time FROM warehouse.etl_watermark ORDER BY layer, table_name', engine)

## 3 · Incremental run
Process the current event-time window. On first run this pulls everything since `2019-10-01`.  
On every subsequent run it only touches data newer than the watermark.


In [ ]:
from accent_fleet.pipeline import incremental_flow
incremental_flow()

In [ ]:
fact_counts_sql = '''
SELECT 'fact_trip'               AS fact, COUNT(*) AS rows FROM warehouse.fact_trip
UNION ALL SELECT 'fact_overspeed',       COUNT(*) FROM warehouse.fact_overspeed
UNION ALL SELECT 'fact_stop',            COUNT(*) FROM warehouse.fact_stop
UNION ALL SELECT 'fact_speed_notification', COUNT(*) FROM warehouse.fact_speed_notification
UNION ALL SELECT 'fact_daily_activity',  COUNT(*) FROM warehouse.fact_daily_activity
'''
pd.read_sql(fact_counts_sql, engine)

### 3.1 Demonstrating idempotency
The core streaming invariant: running the same window twice must produce the same output. If this cell shows the row count unchanged, the pipeline is safe to retry under failure.


In [ ]:
before = pd.read_sql('SELECT COUNT(*) AS n FROM warehouse.fact_trip', engine).iloc[0]['n']
incremental_flow()  # same window again
after  = pd.read_sql('SELECT COUNT(*) AS n FROM warehouse.fact_trip', engine).iloc[0]['n']
print(f'Rows before second run: {before:,}')
print(f'Rows after  second run: {after:,}')
assert abs(after - before) < 100, 'Idempotency broken!'
print('Idempotent: OK')

## 4 · Feature mart
The ML-ready feature table (`marts.v_ml_features_driver_behavior`). Each row = one device for one month.


In [ ]:
features = pd.read_sql('''
    SELECT * FROM marts.v_ml_features_driver_behavior
    WHERE year_month >= '2025-01'
    ORDER BY year_month DESC, total_trips DESC
    LIMIT 20
''', engine)
features

In [ ]:
# Feature-group sanity: show how many non-null values each feature has
features.describe().T.loc[:, ['count','mean','std','min','50%','max']]

## 5 · Risk scoring


In [ ]:
risk = pd.read_sql('''
    SELECT tenant_id, device_id, latest_month,
           trips_3m, distance_3m,
           overspeed_3m, severe_overspeed_3m,
           risk_score, risk_category
    FROM marts.v_device_risk_profile
    ORDER BY risk_score DESC
    LIMIT 25
''', engine)
risk

In [ ]:
# Category distribution
pd.read_sql('''
    SELECT risk_category, COUNT(*) AS devices
    FROM marts.v_device_risk_profile
    GROUP BY risk_category
    ORDER BY CASE risk_category
        WHEN 'critical' THEN 1
        WHEN 'high' THEN 2
        WHEN 'moderate' THEN 3
        WHEN 'low' THEN 4
    END
''', engine)

### 5.1 Python reference scorer matches the SQL view
Pull one row of features and score it both ways.


In [ ]:
from accent_fleet.features import load_risk_scorer
scorer = load_risk_scorer()

# Pick a device with decent history
sample = pd.read_sql('''
    SELECT * FROM marts.mart_device_monthly_behavior
    WHERE total_trips >= 20
    ORDER BY overspeed_per_100km DESC NULLS LAST
    LIMIT 1
''', engine).iloc[0].to_dict()

python_score = scorer.score(sample)
print(f'Python scorer:   {python_score} ({scorer.categorize(python_score)})')
print(f'Tenant/Device:   {sample["tenant_id"]}/{sample["device_id"]}')


## 6 · Stream simulation
This cell simulates a real-time stream by replaying the last hour of staging.path row-by-row through the cleaning rule engine. Same code path that the Kafka consumer will use in production.


In [ ]:
from datetime import datetime, timedelta
from accent_fleet.cleaning import load_rule_engine

# Pull the last hour of trips, batch them like a stream would
last_hour_sql = '''
SELECT tenant_id, device_id, begin_path_time,
       path_duration, distance_driven, max_speed, fuel_used
FROM staging.path
WHERE begin_path_time >= (SELECT MAX(begin_path_time) - INTERVAL '1 hour' FROM staging.path)
ORDER BY begin_path_time
'''
live_df = pl.read_database(last_hour_sql, engine)
print(f'Replaying {live_df.height} trips...')

engine_rules = load_rule_engine()

# Replay in 50-event micro-batches
BATCH = 50
totals = {'in': 0, 'out': 0, 'rejected_by_rule': {}}
for i in range(0, live_df.height, BATCH):
    chunk = live_df.slice(i, BATCH)
    cleaned, result = engine_rules.apply(chunk, table='path')
    totals['in']  += result.total_in
    totals['out'] += result.total_out
    for rule_id, n in result.rejected_by_rule.items():
        totals['rejected_by_rule'][rule_id] = totals['rejected_by_rule'].get(rule_id, 0) + n

print(f'\nStream replay summary:')
print(f'  in  : {totals["in"]}')
print(f'  out : {totals["out"]}')
print(f'  rejected by rule : {totals["rejected_by_rule"]}')

## 7 · Quality & freshness


In [ ]:
from accent_fleet.monitoring import run_validation_suite

report = run_validation_suite()
print(f'All passed: {report.all_passed}')
pd.DataFrame(report.checks)

In [ ]:
# Freshness: how far behind is the warehouse vs staging?
pd.DataFrame(report.freshness).T

## 8 · What changes for streaming

When Accent stands up Kafka, you change **exactly three things**:

1. Producer side — publish staging-shaped JSON to the `fleet.*.v1` topics.
2. Swap `BatchStagingSource` → `StreamKafkaSource` in `pipeline/flow_stream.py` (already wired).
3. Run `scripts/run_streaming.py` instead of / alongside the cron-driven `run_batch.py --mode incremental`.

Everything else — the cleaning rules, the feature definitions, the risk score, the UPSERT semantics, the watermark — is unchanged. That is the payoff of this refactor.
